<a href="https://colab.research.google.com/github/yu-hidaka/AD-DIFFI/blob/develop/examples/Real_World_Analysis/Real_World_Analysis_Breast_cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Real-World Analysis: Breast cancer (Table S3)**
**Project: Adjust DIFFI (AD-DIFFI): Robust Feature Importance for Mixed-Type Data**


## Cell 1: Environment Setup

In [ ]:
# 1. Install required libraries
!pip install -q lifelines scikit-learn pandas numpy matplotlib tabulate

import os
import sys
import importlib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# --- 2. Repository Setup ---
REPO_NAME = "AD-DIFFI"
REPO_URL = "https://github.com/yu-hidaka/AD-DIFFI.git"
BRANCH = "develop"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_NAME}...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print(f"{REPO_NAME} already exists.")

# --- 3. Path & Structure Configuration ---
project_root = os.path.abspath(REPO_NAME)
project_src = os.path.join(project_root, 'src')

conflict_file = os.path.join(project_root, 'ad_diffi.py')
if os.path.exists(conflict_file):
    print(f"Renaming {conflict_file} to avoid import conflict.")
    os.rename(conflict_file, os.path.join(project_root, 'ad_diffi_backup.py'))

if project_src not in sys.path:
    sys.path.insert(0, project_src)
    print(f"Added to path: {project_src}")

# --- 4. File Fixes (core.py / benchmarks.py) ---
core_file_path = os.path.join(project_src, 'ad_diffi', 'core.py')
benchmarks_file_path = os.path.join(project_src, 'ad_diffi', 'benchmarks.py')

if os.path.exists(benchmarks_file_path):
    with open(benchmarks_file_path, 'r') as f:
        lines = f.readlines()
    if lines and '%%writefile' in lines[0]:
        print(f"Removing '%%writefile' from {benchmarks_file_path}")
        with open(benchmarks_file_path, 'w') as f:
            f.writelines(lines[1:])

if os.path.exists(core_file_path):
    with open(core_file_path, 'r') as f:
        core_content = f.read()
    if "def calculate_diffi_scores" not in core_content:
        print(f"Adding 'calculate_diffi_scores' to {core_file_path}")
        add_code = """
def calculate_diffi_scores(iforest, X, feature_names):
    n_samples, n_features = X.shape
    diffi_scores = np.zeros(n_features)
    initial_scores = iforest.decision_function(X)
    for i in range(n_features):
        X_perturbed = X.copy()
        np.random.shuffle(X_perturbed[:, i])
        perturbed_scores = iforest.decision_function(X_perturbed)
        diffi_scores[i] = np.sum(initial_scores - perturbed_scores)
    return diffi_scores
"""
        with open(core_file_path, 'a') as f:
            f.write(add_code)

# --- 5. Execute Import ---
try:
    for mod in list(sys.modules.keys()):
        if mod.startswith('ad_diffi'):
            del sys.modules[mod]

    import ad_diffi
    from ad_diffi.core import calculate_ad_diffi_zscore, diffi_ib_binary_rso, calculate_diffi_scores
    from ad_diffi.utils import download_adbench_dataset, get_feature_metadata
    from ad_diffi.benchmarks import run_diffi_comparison_benchmark

    print("\n[SUCCESS] All ad_diffi modules imported successfully from src/ad_diffi.")
    print(f"Imported from: {ad_diffi.__file__}")

except ImportError as e:
    print(f"\n[ERROR] Import failed: {e}")
    print("If still failing, try 'Runtime -> Restart session' and run this cell again.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.9 MB/s eta 0:00:00
Cloning AD-DIFFI...
Cloning into 'AD-DIFFI'...
remote: Enumerating objects: 314, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 314 (delta 91), reused 11 (delta 11), pack-reused 133 (from 1)
Receiving objects: 100% (314/314), 884.11 KiB | 24.56 MiB/s, done.
Resolving deltas: 100% (159/159), done.
Renaming /content/AD-DIFFI/ad_diffi.py to avoid import conflict.
Added to path: /content/AD-DIFFI/src
Adding 'calculate_diffi_scores' to /content/AD-DIFFI/src/ad_diffi/core.py

[SUCCESS] All ad_diffi modules imported successfully from src/ad_diffi.
Imported from: /content/AD-DIFFI/src/ad_diffi/__init__.py


## Cell 2: Theoretical Framework

## Chapter 5: Implementing AD-DIFFI — RSO and Standardization

### 5.1. Research Objective: Solving the Binary Bias
The primary objective of this analysis is to validate how the **AD-DIFFI** framework corrects the "Score Reversal" phenomenon. In the original DIFFI, highly efficient binary signals were paradoxically penalized because they isolated anomalies too quickly to accumulate "Inlier" scores in deeper nodes.

To resolve this structural bias, we implement and evaluate two primary mechanisms:

#### 5.1.1. Root-Split-Only (RSO) Constraint
Binary features are inherently limited in their splitting capacity (only 0 or 1). When a binary feature is used in deeper nodes of an Isolation Tree, it often represents random noise rather than a meaningful structural split. The **RSO Constraint** restricts the evaluation of binary features exclusively to the **root split (Depth 0)**. This prevents "noise accumulation" in the denominator and ensures that binary features are only rewarded for their primary discriminative power.

#### 5.1.2. Z-score Standardization
Binary and continuous features have fundamentally different probability distributions in an Isolation Forest. To compare them fairly, we establish a **Noise Baseline** by running the model on pure noise datasets.

We calculate the mean ($\mu_{noise}$) and standard deviation ($\sigma_{noise}$) of scores for each feature type separately. Every raw score is then converted into a **Z-score**:

$$Z = \frac{Score_{raw} - \mu_{noise}}{\sigma_{noise}}$$

This transformation creates a unified, zero-centered scale where any score above 0 indicates a signal stronger than random noise, regardless of the data type.

## Cell 3: Data Loading & Metadata Identification

In [ ]:
import pandas as pd
import numpy as np
import os
import subprocess
from pathlib import Path
from google.colab import files

# =============================================================================
# 1. CONFIGURATION & SETUP
# =============================================================================
RANDOM_SEED = 42
DATASET_NAME = "breast_cancer"
LABEL_COLUMN = "Outlier_label"

np.random.seed(RANDOM_SEED)

# =============================================================================
# 2. DATA ACQUISITION & PREPROCESSING
# =============================================================================
def get_breast_cancer_data():
    data_dir = Path('/tmp/ad_diffi_data')
    data_dir.mkdir(exist_ok=True, parents=True)
    csv_file = data_dir / 'Breast_Cancer.csv'

    if not csv_file.exists():
        print("Kaggle setup required for Breast Cancer dataset...")
        uploaded = files.upload()
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
            f.write(uploaded['kaggle.json'])
        os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

        # Download from: https://www.kaggle.com/datasets/reihanenamdari/breast-cancer
        subprocess.run(['kaggle', 'datasets', 'download', '-d', 'reihanenamdari/breast-cancer', '-p', str(data_dir), '--unzip'], check=True)

    df_raw = pd.read_csv(csv_file)

    # Labeling: 'Dead' as Outlier ('o'), 'Alive' as Normal ('n')
    df_processed = df_raw.copy()
    df_processed[LABEL_COLUMN] = np.where(df_processed['Status'] == 'Dead', 'o', 'n')

    # Prepare Survival Target Columns (Required for Cox Evaluation)
    # Status_num: 1 if Dead (Event), 0 if Alive (Censored)
    df_processed['Status_num'] = (df_processed['Status'] == 'Dead').astype(int)
    # 'Survival Months' is kept as is for duration

    # Identify categorical features for One-Hot Encoding
    # We exclude outcome variables ('Status', 'Outlier_label') from encoding
    cat_features = df_processed.select_dtypes(include=['object']).columns.tolist()
    exclude_from_encoding = [LABEL_COLUMN, 'Status']
    cat_features = [c for c in cat_features if c not in exclude_from_encoding]

    # One-hot encoding with drop_first=True to handle multi-collinearity
    df_processed = pd.get_dummies(df_processed, columns=cat_features, drop_first=True, dtype=int)

    # Drop the original 'Status' string column as it's now redundant
    df_processed = df_processed.drop(columns=['Status'], errors='ignore')

    return df_processed

# Load dataset
df = get_breast_cancer_data()

# =============================================================================
# 3. METADATA & ROBUST STATISTICS
# =============================================================================
# Define Target Columns that must be excluded from the feature set (X)
# 'Survival Months' and 'Status_num' are only for evaluation (y)
TARGET_COLS = [LABEL_COLUMN, 'Survival Months', 'Status_num']
feature_names = [col for col in df.columns if col not in TARGET_COLS]

# Convert labels to binary 1/0 for Anomaly Detection evaluation
y = (df[LABEL_COLUMN] == 'o').astype(int).values

# Robust counting based on unique values in the DataFrame
n_bin = 0
n_cont = 0
for col in feature_names:
    if df[col].nunique() <= 2:
        n_bin += 1
    else:
        n_cont += 1

# =============================================================================
# 4. FINAL FORMATTED DISPLAY
# =============================================================================
print(f"--- DATASET STATISTICS: {DATASET_NAME.upper()} ---")
print(f"Total Samples:   {len(df)}")
print(f"Total Features:  {len(feature_names)}")
print(f" - Continuous Features: {n_cont}")
print(f" - Binary Features:     {n_bin}")
print(f"Anomaly Ratio:          {y.mean():.2%}")

print(f"\n[Feature Type Breakdown - First 10 Items]")
for i in range(min(10, len(feature_names))):
    t_label = "cont" if df[feature_names[i]].nunique() > 2 else "bin "
    print(f" - {feature_names[i]:<25}: {t_label}")

print("\n--- Feature Preview (First 5 Rows) ---")
try:
    print(df[feature_names].head().to_markdown(index=False))
except Exception:
    print(df[feature_names].head().to_string(index=False))

--- DATASET STATISTICS: BREAST_CANCER ---
Total Samples:   4024
Total Features:  28
 - Continuous Features: 4
 - Binary Features:     24
Anomaly Ratio:          15.31%

[Feature Type Breakdown - First 10 Items]
 - Age                      : cont
 - Tumor Size               : cont
 - Regional Node Examined   : cont
 - Reginol Node Positive    : cont
 - Race_Other               : bin 
 - Race_White               : bin 
 - Marital Status_Married   : bin 
 - Marital Status_Separated : bin 
 - Marital Status_Single    : bin 
 - Marital Status_Widowed   : bin 

--- Feature Preview (First 5 Rows) ---
|   Age |   Tumor Size |   Regional Node Examined |   Reginol Node Positive |   Race_Other |   Race_White |   Marital Status_Married |   Marital Status_Separated |   Marital Status_Single  |   Marital Status_Widowed |   T Stage _T2 |   T Stage _T3 |   T Stage _T4 |   N Stage_N2 |   N Stage_N3 |   6th Stage_IIB |   6th Stage_IIIA |   6th Stage_IIIB |   6th Stage_IIIC |   differentiate_Poorly diffe

## Cell 4: AD-DIFFI Implementation & Comparison Logic

In [ ]:
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

# =============================================================================
# 1. SETUP & IMPORTS
# =============================================================================
if '/content/AD-DIFFI/src' not in sys.path:
    sys.path.insert(0, '/content/AD-DIFFI/src')

try:
    from ad_diffi.core import (
        calculate_ad_diffi_zscore,
        calculate_diffi_scores,
        diffi_ib_binary_rso,
        get_noise_baselines
    )
    from ad_diffi.benchmarks import (
        evaluate_feature_set_auc,
        evaluate_cox_cindex,
        get_top_k_features
    )
    print("[SUCCESS] AD-DIFFI core and benchmark modules loaded.")
except ImportError as e:
    print(f"[ERROR] Failed to import modules: {e}")

# =============================================================================
# 2. ANALYSIS CONFIGURATION
# =============================================================================
IF_PARAMS = {
    "n_estimators": 100,
    "max_samples": 512,
    "contamination": 0.1531,  # Breast Cancer anomaly ratio
}
N_ITER_NOISE = 20
N_ITER_MAIN = 20
TOP_K = 6
DATASET_NAME = "breast_cancer"

# Feature type mapping (using feature_names from Cell 3)
feature_types = {i: ("cont" if df[f].nunique() > 2 else "bin") for i, f in enumerate(feature_names)}

# =============================================================================
# 3. CORE EXECUTION: AD-DIFFI ANALYSIS
# =============================================================================
print("=" * 80)
print(f"Running Integrated Analysis for: {DATASET_NAME.upper()}")
print("=" * 80)

X_values = df[feature_names].values
X_scaled = StandardScaler().fit_transform(X_values)

# --- Step A: Get Noise Baselines (Automated via core.py) ---
print(f"Establishing noise baseline over {N_ITER_NOISE} iterations...")
c_mu, c_sigma, b_mu, b_sigma = get_noise_baselines(
    X_dim=X_scaled.shape[1],
    feature_types=feature_types,
    if_params=IF_PARAMS,
    n_iter=N_ITER_NOISE
)

noise_baselines = {
    'cont': {'mean': c_mu, 'sd': c_sigma},
    'bin': {'mean': b_mu, 'sd': b_sigma}
}

# --- Step B: Main Importance Loop ---
print(f"Calculating feature importance over {N_ITER_MAIN} iterations...")
orig_acc, rso_acc = [], []
for i in range(N_ITER_MAIN):
    model = IsolationForest(**IF_PARAMS, random_state=i).fit(X_scaled)
    orig_acc.append(calculate_diffi_scores(model, X_scaled, feature_names))
    rso_acc.append(diffi_ib_binary_rso(model, X_scaled, feature_types))

mean_orig = np.mean(orig_acc, axis=0)
mean_rso_raw = np.mean(rso_acc, axis=0)
mean_rso_z = calculate_ad_diffi_zscore(mean_rso_raw, feature_types, noise_baselines=noise_baselines)

# Create Comparison Report
compare_df = pd.DataFrame({
    "Feature": feature_names,
    "Type": [feature_types[i] for i in range(len(feature_names))],
    "Original_DIFFI": np.round(mean_orig, 4),
    "AD_DIFFI_RSO_Z": np.round(mean_rso_z, 4),
})
compare_df["Rank_Original"] = compare_df["Original_DIFFI"].rank(ascending=False).astype(int)
compare_df["Rank_AD_DIFFI"] = compare_df["AD_DIFFI_RSO_Z"].rank(ascending=False).astype(int)
compare_df["Rank_Change"] = compare_df["Rank_Original"] - compare_df["Rank_AD_DIFFI"]

print("\n### Feature Importance Comparison ###")
print(compare_df.sort_values("Rank_AD_DIFFI").to_markdown(index=False))

# =============================================================================
# 4. PERFORMANCE EVALUATION (Using benchmarks.py)
# =============================================================================
print("\n" + "=" * 80)
print("Performance Evaluation: IF AUC (Unsupervised) & Cox C-index (Survival)")
print("=" * 80)

# Identify Top-K sets
top_orig = get_top_k_features(compare_df, "Original_DIFFI", k=TOP_K)
top_ad = get_top_k_features(compare_df, "AD_DIFFI_RSO_Z", k=TOP_K)

# Check for survival columns presence
if 'Survival Months' in df.columns and 'Status_num' in df.columns:
    evaluation_tasks = [
        ("All Features", feature_names),
        (f"Original Top-{TOP_K}", top_orig),
        (f"AD-DIFFI Top-{TOP_K}", top_ad)
    ]

    results = []
    for label, f_list in evaluation_tasks:
        # Calculate IF AUC (Anomaly Detection Accuracy)
        if_auc = evaluate_feature_set_auc(df, f_list, target=y, model_type='IF', params=IF_PARAMS)
        # Calculate Cox C-index (Clinical Survival Accuracy)
        c_index = evaluate_cox_cindex(df, f_list, time_col='Survival Months', event_col='Status_num')
        results.append([label, if_auc, c_index])

    # Summary Table
    summary_df = pd.DataFrame(results, columns=["Method", "IF AUC", "Cox C-index"])
    print(f"\nSummary Table ({DATASET_NAME.upper()}):")
    print(summary_df.to_markdown(index=False))
else:
    print("[ERROR] Survival columns ('Survival Months' or 'Status_num') are missing.")

# Stability Metrics
rho, _ = spearmanr(compare_df["Rank_Original"], compare_df["Rank_AD_DIFFI"])
print(f"\nRank Stability (Spearman ρ): {rho:.4f}")
print(f"Mean |RankΔ|: {compare_df['Rank_Change'].abs().mean():.3f}")

[SUCCESS] AD-DIFFI core and benchmark modules loaded.
Running Integrated Analysis for: BREAST_CANCER
Establishing noise baseline over 20 iterations...


/content/AD-DIFFI/src/ad_diffi/core.py:102: RuntimeWarning: invalid value encountered in divide
  fi_outliers = np.where(counter_outliers_ib > 0, cfi_outliers_ib / counter_outliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:103: RuntimeWarning: invalid value encountered in divide
  fi_inliers = np.where(counter_inliers_ib > 0, cfi_inliers_ib / counter_inliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:102: RuntimeWarning: invalid value encountered in divide
  fi_outliers = np.where(counter_outliers_ib > 0, cfi_outliers_ib / counter_outliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:103: RuntimeWarning: invalid value encountered in divide
  fi_inliers = np.where(counter_inliers_ib > 0, cfi_inliers_ib / counter_inliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:102: RuntimeWarning: invalid value encountered in divide
  fi_outliers = np.where(counter_outliers_ib > 0, cfi_outliers_ib / counter_outliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:103: RuntimeWarning: invalid 

Calculating feature importance over 20 iterations...


/content/AD-DIFFI/src/ad_diffi/core.py:102: RuntimeWarning: invalid value encountered in divide
  fi_outliers = np.where(counter_outliers_ib > 0, cfi_outliers_ib / counter_outliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:103: RuntimeWarning: invalid value encountered in divide
  fi_inliers = np.where(counter_inliers_ib > 0, cfi_inliers_ib / counter_inliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:102: RuntimeWarning: invalid value encountered in divide
  fi_outliers = np.where(counter_outliers_ib > 0, cfi_outliers_ib / counter_outliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:103: RuntimeWarning: invalid value encountered in divide
  fi_inliers = np.where(counter_inliers_ib > 0, cfi_inliers_ib / counter_inliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:102: RuntimeWarning: invalid value encountered in divide
  fi_outliers = np.where(counter_outliers_ib > 0, cfi_outliers_ib / counter_outliers_ib, 0)
/content/AD-DIFFI/src/ad_diffi/core.py:103: RuntimeWarning: invalid 


### Feature Importance Comparison ###
| Feature                             | Type   |   Original_DIFFI |   AD_DIFFI_RSO_Z |   Rank_Original |   Rank_AD_DIFFI |   Rank_Change |
|:------------------------------------|:-------|-----------------:|-----------------:|----------------:|----------------:|--------------:|
| Age                                 | cont   |           1.5784 |           2.5614 |              20 |               1 |            19 |
| Tumor Size                          | cont   |           9.0063 |           2.3064 |               5 |               2 |             3 |
| Reginol Node Positive               | cont   |          16.7363 |           2.2297 |               1 |               3 |            -2 |
| Regional Node Examined              | cont   |           2.3038 |           2.1699 |              16 |               4 |            12 |
| A Stage_Regional                    | bin    |           0.6429 |           1.2661 |              21 |               5 |     